In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy


In [3]:
import pandas as pd

!wget https://raw.githubusercontent.com/LIRNEasia/MisinformationCorpusSinhala/main/Corpus.csv -O corpus.csv

df = pd.read_csv("corpus.csv", encoding='latin1')
df.head()

--2025-12-26 14:10:49--  https://raw.githubusercontent.com/LIRNEasia/MisinformationCorpusSinhala/main/Corpus.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6572668 (6.3M) [application/octet-stream]
Saving to: ‘corpus.csv’

corpus.csv          100%[===================>]   6.27M  --.-KB/s    in 0.07s   

2025-12-26 14:10:49 (92.0 MB/s) - ‘corpus.csv’ saved [6572668/6572668]



,Unnamed: 0,X1,domain,datestamp,type,content,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,5418,652,bbc.com/sinhala,2020-04-03 00:00:00,UNCERTAIN,??????????? ?? ?? ??? ????? 200?? ???? ???????...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5420,2077,gosip-lankanews.com,2020-10-07 00:00:00,CREDIBLE,????? ???????????????? ?????? ?????????? ?????...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5422,3476,adaderana.lk,2020-11-03 00:00:00,CREDIBLE,???? ?????? ???????????? ??? ????? ??? ????? ?...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5424,2675,ravaya.lk,2020-09-19 00:00:00,UNCERTAIN,?????? ??? ???? ???????? ??????????? ????? ???...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5426,3635,anidda.lk,2020-10-04 00:00:00,UNCERTAIN,?????? ???????????? ????????? ???? ??? ???????...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df = df.rename(columns={"content": "text", "type": "label"})

df["label"] = df["label"].replace({
    "CREDIBLE": 0,
    "FALSE": 1,
    "PARTIAL": 1,
    "UNCERTAIN": 1
})

df = df[["text", "label"]].dropna()
df["label"].value_counts()

/tmp/ipython-input-2833683401.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["label"] = df["label"].replace({


,count
label,
1,1997
0,1003


In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["label"]
)


In [7]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df.reset_index(drop=True))


In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-large"   # 🔥 FIX 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256   # 🔥 FIX 2 (can try 384 if GPU allows)
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset  = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [9]:
import torch
from torch.nn import CrossEntropyLoss
from transformers import AutoModelForSequenceClassification

class_counts = df["label"].value_counts().sort_index()
total = class_counts.sum()

class_weights = torch.tensor(
    [total / class_counts[0], total / class_counts[1]],
    dtype=torch.float
).cuda()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.classifier.loss_fct = CrossEntropyLoss(weight=class_weights)


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


In [12]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./xlmr_sinhala_fixed",

    eval_strategy="epoch",     # ✅ FIX
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,

    num_train_epochs=5,
    weight_decay=0.01,

    fp16=True,
    logging_steps=50,

    report_to="none"
)


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.629800,0.643229,0.666667,0.800000
2,0.676000,0.641439,0.666667,0.800000
3,0.616100,0.644043,0.666667,0.800000
4,0.671500,0.637697,0.666667,0.800000


KeyboardInterrupt: 